# Analysis for PsyNamic Manuscript

In [1]:
import os
import pandas as pd
import sys
sys.path.append(os.path.abspath(".."))

def get_df_from_dir(file_dir: str):
    files = [f for f in os.listdir(file_dir) if os.path.isfile(os.path.join(file_dir, f))]
    df_list = []
    for file in files:
        file_path = os.path.join(file_dir, file)
        df = pd.read_csv(file_path)
        df_list.append(df)
    return pd.concat(df_list, ignore_index=True)


## Analysis of data fetching pipeline
### PubMed Fetch Results
How many articles were retrieved from PubMed via API until 31.12.2025?

In [2]:
file = "../data/pubmed_fetch_results/pubmed_results_20240105_20260320_00-00-10.csv"
df = pd.read_csv(file)
print(f'Total articles retrieved: {len(df)}')
df_2025_pubmed_fetch = df[df["entrez_year"] < 2026]
print(f'Total articles retrieved until 2025: {len(df_2025_pubmed_fetch)}')
print(f'Total articles without abstract until 2025: {len(df_2025_pubmed_fetch[df_2025_pubmed_fetch["abstract"].isna()])}')


Total articles retrieved: 1323
Total articles retrieved until 2025: 1172
Total articles without abstract until 2025: 15


How many articles were retrieved from PubMed via API in 2026?

In [3]:
df_2026_pubmed_fetch = df[df["entrez_year"] == 2026]
print(f'Total articles retrieved in 2026: {len(df_2026_pubmed_fetch)}')
print(f'Total articles without abstract in 2026: {len(df_2026_pubmed_fetch[df_2026_pubmed_fetch["abstract"].isna()])}')

Total articles retrieved in 2026: 151
Total articles without abstract in 2026: 1


Are there any articles in PubMed data that are already in the ASreview retrieved dataset?

In [4]:
as_file = "../data/manual/studies_relevant_with_info_20240101_00-00-00.csv"
df_as_relevant = pd.read_csv(as_file)
pubmed_ids = set(df_as_relevant['pubmed_id'].dropna().astype(int).tolist())
pubmed_ids_api = set(df_2025_pubmed_fetch['pubmed_id'].dropna().astype(int).tolist())

overlap_ids = pubmed_ids.intersection(pubmed_ids_api)
print(f'Number of articles in ASreview dataset that are also in PubMed API dataset: {len(overlap_ids)}')
# print title and year of overlapping articles
overlap_df = df_as_relevant[df_as_relevant['pubmed_id'].isin(overlap_ids)]
print(overlap_df[['pubmed_id', 'title', 'year']]) 
df_overlap = df_2025_pubmed_fetch[df_2025_pubmed_fetch['pubmed_id'].isin(overlap_ids)]
print(df_overlap[['pubmed_id', 'title', 'entrez_year']])

Number of articles in ASreview dataset that are also in PubMed API dataset: 1
      pubmed_id                                              title  year
540  38283689.0  Intramuscular ketamine vs. midazolam for rapid...  2024
      pubmed_id                                              title  \
1262   38283689  Intramuscular ketamine vs. midazolam for rapid...   

      entrez_year  
1262         2024  


Are there any duplicates in the 2025 PubMed API data? Or in the ASreview retrieved dataset?

In [5]:
duplicates = df_2025_pubmed_fetch[df_2025_pubmed_fetch.duplicated(subset=['pubmed_id'], keep=False)]
print(f'Number of duplicate articles in 2025 PubMed API dataset: {len(duplicates)}')

# Duplicates not counting nan
duplicates_no_nan = df_2025_pubmed_fetch[df_2025_pubmed_fetch.duplicated(subset=['pubmed_id'], keep=False) & df_2025_pubmed_fetch['pubmed_id'].notna()]
print(f'Number of duplicate articles in 2025 PubMed API dataset (excluding NaN pubmed_ids): {len(duplicates_no_nan)}')


Number of duplicate articles in 2025 PubMed API dataset: 0
Number of duplicate articles in 2025 PubMed API dataset (excluding NaN pubmed_ids): 0


### Relevant Studies from PubMed API
How many articles were classified relevant from PubMed API data until 31.12.2025?

In [6]:
rel_file = "../data/relevant_studies/studies_20260320_00-09-37.csv"
rel_df = pd.read_csv(rel_file)
rel_df_2025 = rel_df[rel_df["entrez_year"] <= 2025]
print(f'Total articles relevant until 2025: {len(rel_df_2025)}')
# without abstracts
print(f'Total articles relevant without abstract until 2025: {len(rel_df_2025[rel_df_2025["abstract"].isna()])}')
# relevant where prediction = 1
df_2025_relevant = rel_df_2025[rel_df_2025["prediction"] == 1]
print(f'Total articles classified relevant until 2025: {len(df_2025_relevant)}')
print(f'Total articles classified irrelevant until 2025: {len(rel_df_2025[rel_df_2025["prediction"] == 0])}')

Total articles relevant until 2025: 1157
Total articles relevant without abstract until 2025: 0
Total articles classified relevant until 2025: 634
Total articles classified irrelevant until 2025: 523


### Total relevant studies up until 31.12.2025


In [7]:
df_2025_relevant, df_as_relevant
merged_df = pd.concat([df_2025_relevant, df_as_relevant], ignore_index=True)
# remove duplicates based on pubmed_id (NaNs should not be considered duplicates)
# keep all rows where pubmed_id is NaN; drop duplicates only among non-NaN ids
mask_drop = merged_df['pubmed_id'].notna() & merged_df.duplicated(subset=['pubmed_id'], keep='first')
merged_df = merged_df[~mask_drop].reset_index(drop=True)

print(f'Total ASreview relevant articles: {len(df_as_relevant)}')
print(f'Total relevant articles from PubMed API until 2025: {len(df_2025_relevant)}')

print(f'Total articles in merged dataset: {len(merged_df)}')


Total ASreview relevant articles: 3313
Total relevant articles from PubMed API until 2025: 634
Total articles in merged dataset: 3946


Do the numbers add up to the PsyNamic Webapp dashboard?

In [8]:
df_2026_relevant = rel_df[(rel_df["entrez_year"] == 2026) & (rel_df["prediction"] == 1)]
print(f'Total articles classified relevant in 2026: {len(df_2026_relevant)}')

len(merged_df) + len(df_2026_relevant)

Total articles classified relevant in 2026: 75


4021

## Analysis of predictions

In [9]:
merged_df.head()

,id,text,prediction,probability,keywords,pubmed_id,pubmed_url,doi,pub_date,entrez_year,...,custom6,custom5,note,custom7,original_publication,custom3,asreview_prior,exported_notes_1,included,asreview_ranking
0,41475562,A systematic review of ketamine and esketamine...,1.0,"[0.0032017924822866917, 0.9967982172966003]","['Depressive disorder', 'Drug administration s...",41475562.0,https://pubmed.ncbi.nlm.nih.gov/41475562/,10.1016/j.jad.2025.121081,2026-04-15,2025.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,41468673,Challenges of enrolling participants with alco...,1.0,"[0.0032391853164881468, 0.9967607855796814]","['Alcohol use disorder', 'Clinical trials', 'E...",41468673.0,https://pubmed.ncbi.nlm.nih.gov/41468673/,10.1016/j.genhosppsych.2025.12.019,2026-01-01,2025.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,41459830,Comparing single- and repeat-dose psilocybin w...,1.0,"[0.002905561588704586, 0.9970943927764893]","['diphenhydramine', 'headache disorders', 'mig...",41459830.0,https://pubmed.ncbi.nlm.nih.gov/41459830/,10.1111/head.70024,2025-12-29,2025.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,41456567,Single fixed-dose intranasal racemic ketamine ...,1.0,"[0.002612984273582697, 0.9973869919776917]","['Acute psychiatry', 'Ketamine', 'Mood disorde...",41456567.0,https://pubmed.ncbi.nlm.nih.gov/41456567/,10.1016/j.psychres.2025.116909,2026-03-01,2025.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,41450831,Acute and post-dosing effects of single-dose p...,1.0,"[0.004768026061356068, 0.9952319264411926]","['acute effects', 'adult psychiatry', 'interpr...",41450831.0,https://pubmed.ncbi.nlm.nih.gov/41450831/,10.3389/fpsyt.2025.1726818,2025-01-01,2025.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [19]:
from components import graphs

class_data = pd.read_csv("../analysis/export/classification_data.csv")
ner_data = pd.read_csv("../analysis/export/ner_data.csv")
study_mapping = pd.read_csv("../analysis/export/study_data.csv")

ids = merged_df['id'].dropna().astype(int).tolist()
# check how many id in class_data and ner_data are in ids
class_ids = class_data['study_id'].unique()
ner_ids = ner_data['study_id'].unique()
print(f'Number of study_ids in classification data: {len(class_ids)}')
print(f'Number of study_ids in NER data: {len(ner_ids)}')

# filter according to ids, "study_id"
filtered_class_data = class_data[class_data['study_id'].isin(ids)]
filtered_ner_data = ner_data[ner_data['study_id'].isin(ids)]
print(f'Number of rows in filtered classification data (filtered): {len(filtered_class_data["study_id"].unique())}')
print(f'Number of rows in filtered NER data (filtered): {len(filtered_ner_data["study_id"].unique())}')


Number of study_ids in classification data: 4020
Number of study_ids in NER data: 1771
Number of rows in filtered classification data (filtered): 3944
Number of rows in filtered NER data (filtered): 1714


In [ ]:
from components.graphs import create_box_plot, bar_chart


